### **Sieć neuronowa MTL na reprezentacji fingerprint - zestaw 2 - Eliminacja**

Wykorzystana reprezentacja: **ECFP4**

Lista endpointów:


1. Half Life (Obach)
2. Clearance Hepatocyte (AstraZeneca)
3. CYP3A4 Inhibition (Veith)
4. VDss (Lombardo)
5. AMES Mutagenicity

Wyniki dla STL:

In [ ]:
!pip install rdkit
!pip install pandas numpy scikit-learn -U
!pip install pytdc --no-dependencies
!pip install fuzzywuzzy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytdc 1.1.15 requires cellxgene-census==1.15.0, which is not installed.
pytdc 1.1.15 requires dataclasses<1.0,>=0.6, which is not installed.
pytdc 1.1.15 requires evaluate==0.4.2, which is not installed.
pytdc 1.1.15 requires gget<1.0.0,>=0.28.4, which is not installed.
pytdc 1.1.15 requires tiledbsoma<2.0.0,>=1.7.2, which is not installed.
pytdc 1.1.15 requires accelerate==0.33.0, but you have accelerate 1.13.0 which is incompatible.
pytdc 1.1.15 requires datasets<2.20.0, but you have datasets 4.0.0 which is incompatible.
pytdc 1.1.15 requires huggingface-hub<1.0,>=0.20.3, but you have huggingface-hub 1.11.0 which is incompatible.
pytdc 1.1.15 requires numpy<2.0.0,>=1.26.4, but you have numpy 2.4.4 which is incompatible.
pytdc 1.1.15 requires pandas<3.0.0,>=2.1.4, but you have pandas 3.0.3 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_folder = "/content/drive/MyDrive/mldd_data" #folder z zapisanymi zestawami danych aby umożliwić porównanie danych na dokładnie tych samych zestawach dla każdego modelu

#data_folder = "/content/drive/MyDrive/MLDD - ADMET/data_splits"

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from tdc.single_pred import ADME
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score, accuracy_score, f1_score

In [ ]:
class Featurizer:
    def __init__(self, y_column, smiles_col='Drug', **kwargs):
        self.y_column = y_column
        self.smiles_col = smiles_col
        self.__dict__.update(kwargs)
    def __call__(self, df):
        raise NotImplementedError()

class ECFPFeaturizer(Featurizer): #dodane zabezpieczenia na Nan etc
    def __init__(self, y_column='Y', radius=2, length=1024, **kwargs):
        self.radius = radius
        self.length = length
        super().__init__(y_column, **kwargs)

    def __call__(self, df):
        fingerprints = []
        labels = []
        for i, row in df.iterrows():
            smiles = row[self.smiles_col]
            mol = Chem.MolFromSmiles(str(smiles)) if pd.notna(smiles) else None

            if mol:
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, self.radius, nBits=self.length)
                fingerprints.append(np.array(fp))
            else:
                # Zamiast pomijać wiersz, wstawiamy wektor zer (zachowuje wyrównanie)
                fingerprints.append(np.zeros(self.length))

            # Pobieramy etykietę jeśli istnieje (dla dummy X wstawiamy NaN)
            labels.append(row[self.y_column] if self.y_column in df.columns else np.nan)

        return np.array(fingerprints), np.array(labels).reshape(-1, 1)

In [ ]:

# DEFINICJA SIECI NEURONOWEJ

class AdmetEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, dropout=0.2):
        super().__init__()
        self.main = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.res_layer = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        x = self.main(x)
        return torch.relu(x + self.res_layer(x))


class MTL_NN_Hybrid(nn.Module):
    def __init__(self, input_dim, reg_tasks, class_tasks, hidden_dim=512):
        """
        reg_tasks: lista nazw zadań regresyjnych (np. ['Lipophilicity', 'Solubility'])
        class_tasks: lista nazw zadań klasyfikacyjnych (np. ['BBB', 'Ames'])
        """
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.reg_tasks = reg_tasks
        self.class_tasks = class_tasks

        # Słowniki głowic dla każdego typu zadania
        #Pozwala zapisać wiele warstw Linear i nazwać je tak, jak nazywają się Twoje zadania.
        self.reg_heads = nn.ModuleDict({
            task: nn.Linear(hidden_dim, 1) for task in reg_tasks
        })
        self.class_heads = nn.ModuleDict({
            task: nn.Linear(hidden_dim, 1) for task in class_tasks
        })

    def forward(self, x):
        shared_features = self.encoder(x) #wspólny enkoder
        results = {}

        # Procesowanie zadań regresyjnych
        for task in self.reg_tasks:
            results[task] = self.reg_heads[task](shared_features)

        # Procesowanie zadań klasyfikacyjnych
        for task in self.class_tasks:
            # results[task] = torch.sigmoid(self.class_heads[task](shared_features))
            results[task] = self.class_heads[task](shared_features)

        #print(results)
        return results #Zwracając słownik:{'Caco2_Wang': 1.2, 'Lipophilicity_AZ': 0.5} masz absolutną pewność, który wynik dotyczy którego badania.

In [ ]:

# --- STL REGRESOR ---
class STL_NN_Regressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1) # Wynik liniowy (dowolna liczba)

    def forward(self, x):
        return self.head(self.encoder(x))

# --- STL KLASYFIKATOR ---
class STL_NN_Classifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # Sigmoid zamienia wynik na prawdopodobieństwo (0-1)
        return torch.sigmoid(self.head(self.encoder(x)))

  #=============================

In [ ]:
def train_MTL_hybrid_wl_sum(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- 1. Normalizacja (osobny skaler dla każdego zadania regresyjnego) ---
    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        # Wyciągamy dane i maskujemy NaN, aby skaler zadziałał
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        # Testowe przesyłamy w oryginale (skalujemy tylko przy ewaluacji dla wygody)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        # Klasyfikacja nie wymaga skalowania
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    # --- 2. Model i Optymalizator ---
    model = MTL_NN_Hybrid(input_dim=1024, reg_tasks=reg_tasks, class_tasks=class_tasks).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none') # 'none' pozwala nam ręcznie obsłużyć NaN
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')

    print("\n--- START TRENINGU (Multi-Task) ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        total_loss = 0

        # Sumujemy straty ze wszystkich zadań, ignorując NaN
        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask]).mean()
                total_loss += loss

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask]).mean()
                total_loss += loss

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")

    # --- 3. Ewaluacja ---
    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            # Odwracamy skalowanie
            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()


            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask]) # <-- Dodaj tę linię
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,  # <-- I tę linię
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [ ]:
def train_MTL_hybrid_uniform_average(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    model = MTL_NN_Hybrid(input_dim=1024, reg_tasks=reg_tasks, class_tasks=class_tasks).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none')
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')

    print("\n--- START TRENINGU (Multi-Task - Uniform Average) ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)

        task_losses = []

        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask]).mean()
                task_losses.append(loss)

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask]).mean()
                task_losses.append(loss)

        if task_losses:
            total_loss = torch.stack(task_losses).mean() # Uniform average
        else:
            total_loss = torch.tensor(0.0, device=device, requires_grad=True)

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")

    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()

            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask])
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [ ]:
def train_MTL_hybrid_uncertainty_weighting(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    model = MTL_NN_Hybrid(input_dim=1024, reg_tasks=reg_tasks, class_tasks=class_tasks).to(device)

    # Learnable parameters for task uncertainty
    log_vars = nn.ParameterDict()
    for task in reg_tasks + class_tasks:
        log_vars[task] = nn.Parameter(torch.zeros(1, device=device)) # Initialize log_sigma_sq to 0

    # Include log_vars in the optimizer
    optimizer = optim.Adam(list(model.parameters()) + list(log_vars.values()), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none')
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')

    print("\n--- START TRENINGU (Multi-Task - Uncertainty Weighting) ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        total_loss = 0

        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask])
                precision = torch.exp(-log_vars[task])
                total_loss += torch.mean(0.5 * precision * loss + 0.5 * log_vars[task])

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask])
                precision = torch.exp(-log_vars[task])
                total_loss += torch.mean(precision * loss + 0.5 * log_vars[task])

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")
            # print(f"    Log_vars: {{ {', '.join([f'{t}: {log_vars[t].item():.2f}' for t in log_vars])} }}")

    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()

            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask])
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [ ]:
import pickle

def load_split_pickle(dataset_name):
    filepath = f"{data_folder}/{dataset_name}_split.pkl"

    with open(filepath, "rb") as f:
        split = pickle.load(f)

    return split["train"], split["test"]

In [ ]:
def print_metrics(metrics, task='classification', weight_loss_func_name=None):
    print(f"\n{'='*40}")
    if weight_loss_func_name: # Check if func_name is provided and not None
        print(f"  Loss Weighting: {weight_loss_func_name}")
        print(f"{'='*40}")
    if task == 'classification':
        print(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}")
        print(f"  F1       : {metrics['test_metrics']['f1']:.4f}")
        print(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}")
    else:
        print(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}")
        print(f"  MAE      : {metrics['test_metrics']['mae']:.4f}")
        print(f"  R²       : {metrics['test_metrics']['r2']:.4f}")
    print(f"{'='*40}\n")


def save_metrics(metrics, dataset_name, filepath, task='classification', weight_loss_func_name=None, endpoint_group_name=None):
    with open(filepath, 'a') as f:
        f.write(f"\n{'='*40}\n")
        f.write(f"Endpoint    : {dataset_name}\n")
        if endpoint_group_name:
            f.write(f"Tasks       : {endpoint_group_name}\n")
        if weight_loss_func_name:
            f.write(f"Loss Weighting: {weight_loss_func_name}\n")
        f.write(f"{'='*40}\n")
        if task == 'classification':
            f.write(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}\n")
            f.write(f"  F1       : {metrics['test_metrics']['f1']:.4f}\n")
            f.write(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}\n")
        else:
            f.write(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}\n")
            f.write(f"  MAE      : {metrics['test_metrics']['mae']:.4f}\n")
            f.write(f"  R²       : {metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"{'='*40}\n")

In [ ]:
import numpy as np
import pandas as pd

def prepare_mtl_data_final(df_list, task_names, featurizer):
    # 1. Zebranie wszystkich unikalnych struktur
    all_drugs = set()
    for df in df_list:
        valid = df['Drug'].dropna().astype(str).unique()
        all_drugs.update(valid)

    # 2. Walidacja cząsteczek przez RDKit przed stworzeniem master_list
    # To zapobiega rozbieżnościom rozmiarów X i y
    safe_master_list = []
    for drug in sorted(list(all_drugs)):
        mol = Chem.MolFromSmiles(drug)
        if mol: safe_master_list.append(drug)

    drug_to_idx = {drug: i for i, drug in enumerate(safe_master_list)}
    n_samples = len(safe_master_list)

    # 3. Generowanie X (tylko dla poprawnych cząsteczek)
    df_temp = pd.DataFrame({'Drug': safe_master_list})
    X_features, _ = featurizer(df_temp)

    # Sprawdzenie czy X zawiera NaN (zabezpieczenie przed błędami featurizera)
    if np.isnan(X_features).any():
        X_features = np.nan_to_num(X_features)

    # 4. Mapowanie etykiet y (z zachowaniem NaN tylko w etykietach!)
    y_dict = {}
    for df, task in zip(df_list, task_names):
        y_vec = np.full((n_samples, 1), np.nan, dtype=np.float32)
        # Tworzymy mapę {Drug: Wynik}
        mapping = dict(zip(df['Drug'].astype(str), df['Y']))

        for drug, val in mapping.items():
            if drug in drug_to_idx and not pd.isna(val):
                y_vec[drug_to_idx[drug]] = val
        y_dict[task] = y_vec

    return X_features, y_dict

# Test1: Half Life (Obach) + Clearance Hepatocyte (AstraZeneca)

Test pierwszy sprawdzający czy klirens wątrobowy (Clearance Hepatocyte) pomoże w predykcji okresu półtrwania (Half Life) - oba zadania są bezpośrednio powiązane mechanistycznie. Sprawdzamy czy MTL da lepsze wyniki niż STL.

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ']
class_tasks = [] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eliminacja_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_halflife, train_clear], reg_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_halflife, test_clear],  reg_tasks, featurizer)


# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

[17:55:39] DEPRECATION WARNING: please use MorganGenerator
[17:55:39] DEPRECATION WARNING: please use MorganGenerator
[17:55:39] DEPRECATION WARNING: please use MorganGenerator
[17:55:39] DEPRECATION WARNING: please use MorganGenerator
[17:55:39] DEPRECATION WARNING: please use MorganGenerator
[17:55:39] DEPRECATION WARNING: please use MorganGenerator
[17:55:39] DEPRECATION WARNING: please use MorganGenerator
[17:55:39] DEPRECATION WARNING: please use MorganGenerator
[17:55:39] DEPRECATION WARNING: please use MorganGenerator
[17:55:39] DEPRECATION WARNING: please use MorganGenerator
[17:55:39] DEPRECATION WARNING: please use MorganGenerator
[17:55:39] DEPRECATION WARNING: please use MorganGenerator
[17:55:39] DEPRECATION WARNING: please use MorganGenerator
[17:55:39] DEPRECATION WARNING: please use MorganGenerator
[17:55:39] DEPRECATION WARNING: please use MorganGenerator
[17:55:39] DEPRECATION WARNING: please use MorganGenerator
[17:55:39] DEPRECATION WARNING: please use MorganGenerat


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 2.4106
  Epoka  20 | Total Loss: 0.7930
  Epoka  40 | Total Loss: 0.2043
  Epoka  60 | Total Loss: 0.0683
  Epoka  80 | Total Loss: 0.0409

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Weighted Loss Sum
  RMSE     : 89.2129
  MAE      : 32.3182
  R²       : 0.4452


Wyniki dla zadania (Regresja): Clearance_Hepatocyte_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 49.3420
  MAE      : 35.1223
  R²       : 0.0427


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.1641
  Epoka  20 | Total Loss: 0.3551
  Epoka  40 | Total Loss: 0.0607
  Epoka  60 | Total Loss: 0.0239
  Epoka  80 | Total Loss: 0.0246

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Uniform Average
  RMSE     : 88.5584
  MAE      : 28.2040
  R²     

# Test2: Half Life (Obach) + Clearance Hepatocyte (AstraZeneca) + CYP3A4 Inhibition (Veith)

Dokładamy zadanie klasyfikacyjne CYP3A4 Inhibition. Sprawdzamy czy zadanie powiązane mechanistycznie z eliminacją (CYP3A4 metabolizuje 50% leków) wnosi dodatkową informację. Pierwszy test mieszany (regresja + klasyfikacja).

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ']
class_tasks = ['CYP3A4_Veith'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eliminacja_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_halflife, train_clear, train_cyp], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_halflife, test_clear, test_cyp],  reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[17:56:03] DEPRECATION WARNING: please use MorganGenerator
[17:56:03] DEPRECATION WARNING: please use MorganGenerator
[17:56:03] DEPRECATION WARNING: please use MorganGenerator
[17:56:03] DEPRECATION WARNING: please use MorganGenerator
[17:56:03] DEPRECATION WARNING: please use MorganGenerator
[17:56:03] DEPRECATION WARNING: please use MorganGenerator
[17:56:03] DEPRECATION WARNING: please use MorganGenerator
[17:56:03] DEPRECATION WARNING: please use MorganGenerator
[17:56:03] DEPRECATION WARNING: please use MorganGenerator
[17:56:03] DEPRECATION WARNING: please use MorganGenerator
[17:56:03] DEPRECATION WARNING: please use MorganGenerator
[17:56:03] DEPRECATION WARNING: please use MorganGenerator
[17:56:03] DEPRECATION WARNING: please use MorganGenerator
[17:56:03] DEPRECATION WARNING: please use MorganGenerator
[17:56:03] DEPRECATION WARNING: please use MorganGenerator
[17:56:03] DEPRECATION WARNING: please use MorganGenerator
[17:5


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 3.1480
  Epoka  20 | Total Loss: 1.3622
  Epoka  40 | Total Loss: 0.5985
  Epoka  60 | Total Loss: 0.4215
  Epoka  80 | Total Loss: 0.3043

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Weighted Loss Sum
  RMSE     : 85.8869
  MAE      : 32.6073
  R²       : 0.4858


Wyniki dla zadania (Regresja): Clearance_Hepatocyte_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 49.1181
  MAE      : 34.8007
  R²       : 0.0513


Wyniki dla zadania (Klasyfikacja): CYP3A4_Veith

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7899
  F1       : 0.7384
  AUROC    : 0.8754


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.0736
  Epoka  20 | Total Loss: 0.3900
  Epoka  40 | Total Loss: 0.1778
  Epoka  60 | Total Loss: 0.1355
  Epoka  80 | Total Loss: 0.1077

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uni

# Test3: Half Life (Obach) + Clearance Hepatocyte (AstraZeneca) + CYP3A4 Inhibition (Veith) + VDss (Lombardo)

Dokładamy VDss - kompletny zestaw eliminacyjny (4 endpointy biologicznie powiązane). Sprawdzamy czy pełne pokrycie domeny daje najlepsze wyniki, czy w pewnym momencie wyniki przestają się poprawiać.

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ', 'VDss_Lombardo']
class_tasks = ['CYP3A4_Veith'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eliminacja_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_halflife, train_clear, train_vdss, train_cyp], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_halflife, test_clear, test_vdss, test_cyp],  reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[17:56:31] DEPRECATION WARNING: please use MorganGenerator
[17:56:31] DEPRECATION WARNING: please use MorganGenerator
[17:56:31] DEPRECATION WARNING: please use MorganGenerator
[17:56:31] DEPRECATION WARNING: please use MorganGenerator
[17:56:31] DEPRECATION WARNING: please use MorganGenerator
[17:56:31] DEPRECATION WARNING: please use MorganGenerator
[17:56:31] DEPRECATION WARNING: please use MorganGenerator
[17:56:31] DEPRECATION WARNING: please use MorganGenerator
[17:56:31] DEPRECATION WARNING: please use MorganGenerator
[17:56:31] DEPRECATION WARNING: please use MorganGenerator
[17:56:31] DEPRECATION WARNING: please use MorganGenerator
[17:56:31] DEPRECATION WARNING: please use MorganGenerator
[17:56:31] DEPRECATION WARNING: please use MorganGenerator
[17:56:31] DEPRECATION WARNING: please use MorganGenerator
[17:56:31] DEPRECATION WARNING: please use MorganGenerator
[17:56:31] DEPRECATION WARNING: please use MorganGenerator
[17:5


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 4.1249
  Epoka  20 | Total Loss: 1.3568
  Epoka  40 | Total Loss: 0.6025
  Epoka  60 | Total Loss: 0.4325
  Epoka  80 | Total Loss: 0.3338

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Weighted Loss Sum
  RMSE     : 86.5987
  MAE      : 30.5807
  R²       : 0.4772


Wyniki dla zadania (Regresja): Clearance_Hepatocyte_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 50.2398
  MAE      : 34.7560
  R²       : 0.0075


Wyniki dla zadania (Regresja): VDss_Lombardo

  Loss Weighting: Weighted Loss Sum
  RMSE     : 8.2449
  MAE      : 5.6692
  R²       : -0.4511


Wyniki dla zadania (Klasyfikacja): CYP3A4_Veith

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7936
  F1       : 0.7383
  AUROC    : 0.8763


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.0350
  Epoka  20 | Total Loss: 0.3332
  Epoka  4

# Test4: Clearance Hepatocyte (AstraZeneca) + CYP3A4 Inhibition (Veith)

Sprawdzamy parę regresja + klasyfikacja - oba zadania bezpośrednio związane z metabolizmem wątrobowym. Jednocześnie porównujemy STL vs MTL.

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Clearance_Hepatocyte_AZ']
class_tasks = ['CYP3A4_Veith'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eliminacja_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_clear, train_cyp], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_clear, test_cyp],  reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[17:56:55] DEPRECATION WARNING: please use MorganGenerator
[17:56:55] DEPRECATION WARNING: please use MorganGenerator
[17:56:55] DEPRECATION WARNING: please use MorganGenerator
[17:56:55] DEPRECATION WARNING: please use MorganGenerator
[17:56:55] DEPRECATION WARNING: please use MorganGenerator
[17:56:55] DEPRECATION WARNING: please use MorganGenerator
[17:56:55] DEPRECATION WARNING: please use MorganGenerator
[17:56:55] DEPRECATION WARNING: please use MorganGenerator
[17:56:55] DEPRECATION WARNING: please use MorganGenerator
[17:56:55] DEPRECATION WARNING: please use MorganGenerator
[17:56:55] DEPRECATION WARNING: please use MorganGenerator
[17:56:55] DEPRECATION WARNING: please use MorganGenerator
[17:56:55] DEPRECATION WARNING: please use MorganGenerator
[17:56:55] DEPRECATION WARNING: please use MorganGenerator
[17:56:55] DEPRECATION WARNING: please use MorganGenerator
[17:56:55] DEPRECATION WARNING: please use MorganGenerator
[17:5


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 1.8660
  Epoka  20 | Total Loss: 0.7482
  Epoka  40 | Total Loss: 0.4629
  Epoka  60 | Total Loss: 0.3282
  Epoka  80 | Total Loss: 0.2023

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Clearance_Hepatocyte_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 48.8426
  MAE      : 33.9159
  R²       : 0.0620


Wyniki dla zadania (Klasyfikacja): CYP3A4_Veith

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7920
  F1       : 0.7341
  AUROC    : 0.8756


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.0145
  Epoka  20 | Total Loss: 0.6002
  Epoka  40 | Total Loss: 0.3070
  Epoka  60 | Total Loss: 0.1990
  Epoka  80 | Total Loss: 0.1355

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Clearance_Hepatocyte_AZ

  Loss Weighting: Uniform Average
  RMSE     : 49.1420
  MAE      : 34.2227
  

# Test5: Half Life (Obach) + CYP3A4 Inhibition (Veith)

Sprawdzamy czy CYP3A4 (klasyfikacja) pomoże regresji Half Life - są to zadania powiązane pośrednio przez metabolizm enzymu CYP3A4.

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Half_Life_Obach']
class_tasks = ['CYP3A4_Veith'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eliminacja_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_halflife, train_cyp], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_halflife, test_cyp],  reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[17:57:18] DEPRECATION WARNING: please use MorganGenerator
[17:57:18] DEPRECATION WARNING: please use MorganGenerator
[17:57:18] DEPRECATION WARNING: please use MorganGenerator
[17:57:18] DEPRECATION WARNING: please use MorganGenerator
[17:57:18] DEPRECATION WARNING: please use MorganGenerator
[17:57:18] DEPRECATION WARNING: please use MorganGenerator
[17:57:18] DEPRECATION WARNING: please use MorganGenerator
[17:57:18] DEPRECATION WARNING: please use MorganGenerator
[17:57:18] DEPRECATION WARNING: please use MorganGenerator
[17:57:18] DEPRECATION WARNING: please use MorganGenerator
[17:57:18] DEPRECATION WARNING: please use MorganGenerator
[17:57:18] DEPRECATION WARNING: please use MorganGenerator
[17:57:18] DEPRECATION WARNING: please use MorganGenerator
[17:57:18] DEPRECATION WARNING: please use MorganGenerator
[17:57:18] DEPRECATION WARNING: please use MorganGenerator
[17:57:18] DEPRECATION WARNING: please use MorganGenerator
[17:5


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 1.8934
  Epoka  20 | Total Loss: 1.2464
  Epoka  40 | Total Loss: 0.5608
  Epoka  60 | Total Loss: 0.3655
  Epoka  80 | Total Loss: 0.2688

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Weighted Loss Sum
  RMSE     : 81.1849
  MAE      : 27.4161
  R²       : 0.5405


Wyniki dla zadania (Klasyfikacja): CYP3A4_Veith

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.8058
  F1       : 0.7555
  AUROC    : 0.8763


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 0.9348
  Epoka  20 | Total Loss: 0.5214
  Epoka  40 | Total Loss: 0.2660
  Epoka  60 | Total Loss: 0.1788
  Epoka  80 | Total Loss: 0.1131

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Uniform Average
  RMSE     : 83.0695
  MAE      : 29.9607
  R²       : 0.518

# Test6: Half Life (Obach) + AMES Mutagenicity

Sprawdzamy łączenie z niepowiązanym biologicznie zadaniem (AMES - mutagenność). Oczekujemy transferu negatywnego.

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Half_Life_Obach']
class_tasks = ['AMES'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eliminacja_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_ames, test_ames = load_split_pickle('AMES')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_halflife, train_ames], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_halflife, test_ames],  reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[17:57:33] DEPRECATION WARNING: please use MorganGenerator
[17:57:33] DEPRECATION WARNING: please use MorganGenerator
[17:57:33] DEPRECATION WARNING: please use MorganGenerator
[17:57:33] DEPRECATION WARNING: please use MorganGenerator
[17:57:33] DEPRECATION WARNING: please use MorganGenerator
[17:57:33] DEPRECATION WARNING: please use MorganGenerator
[17:57:33] DEPRECATION WARNING: please use MorganGenerator
[17:57:33] DEPRECATION WARNING: please use MorganGenerator
[17:57:33] DEPRECATION WARNING: please use MorganGenerator
[17:57:33] DEPRECATION WARNING: please use MorganGenerator
[17:57:33] DEPRECATION WARNING: please use MorganGenerator
[17:57:33] DEPRECATION WARNING: please use MorganGenerator
[17:57:33] DEPRECATION WARNING: please use MorganGenerator
[17:57:33] DEPRECATION WARNING: please use MorganGenerator
[17:57:33] DEPRECATION WARNING: please use MorganGenerator
[17:57:33] DEPRECATION WARNING: please use MorganGenerator
[17:5


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 2.1433
  Epoka  20 | Total Loss: 1.3141
  Epoka  40 | Total Loss: 0.6620
  Epoka  60 | Total Loss: 0.3310
  Epoka  80 | Total Loss: 0.1388

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Weighted Loss Sum
  RMSE     : 81.1626
  MAE      : 27.7337
  R²       : 0.5408


Wyniki dla zadania (Klasyfikacja): AMES

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.8239
  F1       : 0.8398
  AUROC    : 0.8907


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 0.9230
  Epoka  20 | Total Loss: 0.4390
  Epoka  40 | Total Loss: 0.1888
  Epoka  60 | Total Loss: 0.0804
  Epoka  80 | Total Loss: 0.0239

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Uniform Average
  RMSE     : 83.7778
  MAE      : 29.4268
  R²       : 0.5107


Wyni

# Test7: Half Life (Obach) + Clearance Hepatocyte (AstraZeneca) + CYP3A4 Inhibition (Veith) + VDss (Lombardo) + AMES Mutagenicity

Pełny zestaw 5 endpointów: 4 powiązane biologicznie + 1 niepowiązany (AMES). Sprawdzamy łączny efekt - czy AMES zaszkodzi pozostałym, czy pełne pokrycie wygrywa.

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ', 'VDss_Lombardo']
class_tasks = ['CYP3A4_Veith', 'AMES'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eliminacja_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')
train_ames, test_ames = load_split_pickle('AMES')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_halflife, train_clear, train_vdss, train_cyp, train_ames], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_halflife, test_clear, test_vdss, test_cyp, test_ames],  reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[17:58:04] DEPRECATION WARNING: please use MorganGenerator
[17:58:04] DEPRECATION WARNING: please use MorganGenerator
[17:58:04] DEPRECATION WARNING: please use MorganGenerator
[17:58:04] DEPRECATION WARNING: please use MorganGenerator
[17:58:04] DEPRECATION WARNING: please use MorganGenerator
[17:58:04] DEPRECATION WARNING: please use MorganGenerator
[17:58:04] DEPRECATION WARNING: please use MorganGenerator
[17:58:04] DEPRECATION WARNING: please use MorganGenerator
[17:58:04] DEPRECATION WARNING: please use MorganGenerator
[17:58:04] DEPRECATION WARNING: please use MorganGenerator
[17:58:04] DEPRECATION WARNING: please use MorganGenerator
[17:58:04] DEPRECATION WARNING: please use MorganGenerator
[17:58:04] DEPRECATION WARNING: please use MorganGenerator
[17:58:04] DEPRECATION WARNING: please use MorganGenerator
[17:58:04] DEPRECATION WARNING: please use MorganGenerator
[17:58:04] DEPRECATION WARNING: please use MorganGenerator
[17:5


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 5.7038
  Epoka  20 | Total Loss: 2.2435
  Epoka  40 | Total Loss: 1.2147
  Epoka  60 | Total Loss: 0.7103
  Epoka  80 | Total Loss: 0.4709

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Weighted Loss Sum
  RMSE     : 85.1582
  MAE      : 34.6881
  R²       : 0.4944


Wyniki dla zadania (Regresja): Clearance_Hepatocyte_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 48.9926
  MAE      : 34.1718
  R²       : 0.0562


Wyniki dla zadania (Regresja): VDss_Lombardo

  Loss Weighting: Weighted Loss Sum
  RMSE     : 8.7279
  MAE      : 5.9127
  R²       : -0.6262


Wyniki dla zadania (Klasyfikacja): CYP3A4_Veith

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7895
  F1       : 0.7393
  AUROC    : 0.8757


Wyniki dla zadania (Klasyfikacja): AMES

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.8116
  F1       : 0.8294
  AUROC    : 